# Interactive Prototype

Goal: Create a functional MVP for field route planning and management.

Stages:
1. Region Input & Sub-Division
2. Automatic Target Search & Routing
3. Manual Adjustment
4. Select & Execute Plans

## Region Input & Sub-Division

- Get region outline
- Divide into work cells
- Tentative plan for depot locations
- 

### Dummy Region Shapefile

At this point, we do not actually have a shapefile of the target region. The following two Jupyter cells will generate one using the outline of the orthophoto we have created. 

Load this as the "shapefile" which will define our working region. 

In [80]:
region_image_path = '../input/IGNORE_Brewster-2024-all-orthophoto-UTM-32613.tif'
region_contour_shapefile = '../input/interactive_proto/region_contour.shp'
region_contour_geojson = '../input/interactive_proto/region_contour.geojson'


region_crs = 32613 # Use this everywhere for consistency
visualization_crs = 4326 # Use this when we need leaflet visualizations
simplification_tolerance = 5


In [88]:
import sys
import geopandas as gpd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from plant_search.load_image import load_image
from plant_search.region_partition import extract_region_contour, simplify_polygon

# image, transform, bounds, image_crs = load_image(region_image_path)

# Get region contour, emulating shapefile input
region_contour_gpd = extract_region_contour(region_image_path)
contour_gdf_projected = region_contour_gpd.to_crs(epsg=region_crs)  # Example: UTM Zone 14N
region_contour = simplify_polygon(contour_gdf_projected.geometry.iloc[0], simplification_tolerance)
# region_contour_gdf = gpd.GeoDataFrame({"geometry": [region_contour]}, crs=region_crs) # Convert to GDF
region_contour_gdf = gpd.GeoDataFrame({"geometry": [region_contour]}) # Convert to GDF

# Save as shape file
region_contour_gdf.to_file(region_contour_shapefile, driver="ESRI Shapefile")

# Prove it can be loaded
loaded_gdf = gpd.read_file(region_contour_shapefile)
print(f"Loaded shapefile CRS: {loaded_gdf.crs}")
print(type(loaded_gdf))

# Convert to GeoJSON
loaded_gdf.to_crs(epsg=4326, inplace=True)
loaded_gdf.to_file(region_contour_geojson, driver="GeoJSON")
print(f"Written geoJSON CRS: {loaded_gdf.crs}")
print(type(loaded_gdf))

Loaded shapefile CRS: EPSG:32613
<class 'geopandas.geodataframe.GeoDataFrame'>
Written geoJSON CRS: EPSG:4326
<class 'geopandas.geodataframe.GeoDataFrame'>


/home/kellan/Documents/Javelina/unsupervised-cv-search/venv/lib/python3.12/site-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


### Create Voronoi Partitioning, Solve for Depots

In [58]:
target_area_acres = 0.5
# target_area_acres = 1.5
# target_area_acres = 2.5

target_area_sqm = target_area_acres * 4046.86
max_iterations = 15

voronoi_partition_filename = '../input/interactive_proto/voronoi_partition.geojson'
voronoi_centroids_filename = '../input/interactive_proto/voronoi_partition.geojson'
# voronoi_partition_packagename = '../input/interactive_proto/voronoi_partition.gpkg'

In [ ]:
from plant_search.region_partition import centroidal_voronoi_tessellation

region_outline_gdf = gpd.read_file(region_contour_shapefile)
simplified_polygon = region_outline_gdf.geometry.iloc[0]
# print(loaded_gdf.crs)

num_points = int(simplified_polygon.area / target_area_sqm) # How many cells to generate

cell_gdf = centroidal_voronoi_tessellation(simplified_polygon, num_points, max_iterations)
print(cell_gdf.crs)

# Create a copy with only the 'geometry' column (Voronoi polygons)
voronoi_gdf = cell_gdf.drop(columns=["cell_centroid"]).copy()
voronoi_gdf.to_crs(visualization_crs, inplace=True)
voronoi_gdf.to_file(voronoi_partition_filename, driver="GeoJSON")

# Create a copy with only the 'cell_centroid' column and set it as the active geometry
centroid_gdf = cell_gdf.copy().drop(columns=["geometry"])
centroid_gdf.set_geometry("cell_centroid", inplace=True)
# centroid_gdf.to_crs(visualization_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.set_crs(visualization_crs)  # Reset the CRS explicitly
centroid_gdf.to_file(voronoi_centroids_filename, driver="GeoJSON")

Reached maximum iterations without full convergence.
EPSG:32613


ValueError: Cannot transform naive geometries.  Please set a crs on the object first.

In [60]:
# import fiona

# # List layers in the GeoPackage
# print(fiona.listlayers(voronoi_partition_packagename))

# # Load each layer and check CRS
# voronoi_gdf = gpd.read_file(voronoi_partition_packagename, layer="voronoi_cells")
# centroid_gdf = gpd.read_file(voronoi_partition_packagename, layer="centroids")

# print(voronoi_gdf.crs)  # Should print the CRS (e.g., EPSG:32614)
# print(centroid_gdf.crs)


In [92]:
import geopandas as gpd
from ipyleaflet import Map, GeoJSON, LayersControl
from shapely.geometry import mapping
import json

# Load Voronoi polygons
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load centroids
with open(voronoi_centroids_filename, "r") as f:
    centroid_data = json.load(f)


# Load the GeoJSON file
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
print(region_contour_data)



# Create a map centered on the region's centroid
region_center = loaded_gdf.geometry.centroid.iloc[0]

m = Map(center=(region_center.y, region_center.x), zoom=13)

# Add the region border to the map
region_layer = GeoJSON(data=region_contour_data, style={'color': 'green', 'fillOpacity': 0.2, 'weight': 3})
m.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(data=voronoi_data, style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2})
m.add_layer(voronoi_layer)

# Add centroids
# centroid_layer = GeoJSON(data=centroid_data, style={'color': 'red', 'radius': 5, 'fillOpacity': 1.0})
# m.add_layer(centroid_layer)

# Add layer control
m.add_control(LayersControl())

# Display the map
m



{'type': 'FeatureCollection', 'name': 'region_contour', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}, 'features': [{'type': 'Feature', 'properties': {'FID': 0}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-103.60354195463582, 30.250675823114307], [-103.60193061465692, 30.250577387216087], [-103.60193104283606, 30.250506613331293], [-103.60137442581158, 30.250486215850458], [-103.60137526390523, 30.25038653684508], [-103.60068742340876, 30.2503079129831], [-103.60004609099659, 30.25047202431435], [-103.59901401121631, 30.25041712589999], [-103.59832880518353, 30.25015309837558], [-103.59797419136667, 30.24945198607267], [-103.5991885592184, 30.247583868146478], [-103.59973176366111, 30.247374848878962], [-103.60047518518951, 30.247135568141697], [-103.60086517298295, 30.247123784423042], [-103.60425409922445, 30.247370780715062], [-103.60483420339533, 30.247515527429865], [-103.60503711335184, 30.247619868860273], [-103.60505367596096, 30.247707

/tmp/ipykernel_16656/4288315622.py:23: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  region_center = loaded_gdf.geometry.centroid.iloc[0]


Map(center=[30.248931657196884, -103.60192091362077], controls=(ZoomControl(options=['position', 'zoom_in_text…